# XIAO Human Tracker Phase 4

Fine-tune YOLO11n on the frozen XIAO study. A uses deterministic resizing and padding. B adds horizontal flips, HSV changes, translation and scale during training. Both start from the same pretrained checkpoint and use the same 60-image validation split.

1. In Colab, choose **Runtime → Change runtime type → GPU**.
2. Upload **datasets/xiao-v1.zip** from the project to **My Drive/Human_Tracker_Phase4** or change the input folder below. You can put the original **models/yolo11n.pt** there too. The notebook fetches the exact official checkpoint if it is absent.
3. Run through the smoke checks. Inspect both training batches, then run the full A and B cells.

The setup and pretrained validation also run on CPU. Fine-tuning requires a GPU runtime.


In [ ]:
%pip install -q ultralytics==8.4.215


In [ ]:
# @title Mount Drive and choose your paths
from pathlib import Path
import os
import sys
from google.colab import drive

drive.mount('/content/drive')
INPUT_FOLDER = '/content/drive/MyDrive/Human_Tracker_Phase4'  # @param {type:"string"}
OUTPUT_FOLDER = '/content/drive/MyDrive/Human_Tracker_Phase4/runs'  # @param {type:"string"}
RUN_GROUP = 'study-v1'  # @param {type:"string"}
BATCH = 8  # @param {type:"integer"}

drive_root = Path('/content/drive/MyDrive').resolve()
inputs = Path(INPUT_FOLDER).resolve()
outputs = Path(OUTPUT_FOLDER).resolve()
if not inputs.is_relative_to(drive_root) or not outputs.is_relative_to(drive_root):
    raise ValueError('Choose input and output folders inside mounted My Drive.')
if not RUN_GROUP or RUN_GROUP in ('.', '..') or Path(RUN_GROUP).name != RUN_GROUP:
    raise ValueError('Choose a single folder name for RUN_GROUP.')
if BATCH < 1:
    raise ValueError('Set BATCH to a positive integer.')
output = outputs / RUN_GROUP
work = Path('/content/xiao-phase4')
helpers = work / 'helpers'
package = helpers / 'human_tracker'
package.mkdir(parents=True, exist_ok=True)
(package / '__init__.py').touch()
sys.path.insert(0, str(helpers))
os.environ.setdefault('YOLO_CONFIG_DIR', str(work / 'ultralytics'))


In [ ]:
%%writefile /content/xiao-phase4/helpers/human_tracker/dataset.py
"""Store native XIAO frames and build a validated YOLO dataset."""

from collections import Counter, defaultdict
import csv
from dataclasses import dataclass, replace
from datetime import datetime, timezone
import hashlib
import io
from itertools import product
import json
import math
import os
from pathlib import Path
import re
import tempfile
from uuid import uuid4
import zipfile

import cv2
import numpy as np


DEFAULT_ROOT = Path('datasets/xiao')
WIDTH, HEIGHT = 320, 240
MAX_FRAME_AGE = 0.5
SPLITS = ('train', 'val', 'test')
MIN_SESSIONS = {'train': 1, 'val': 2, 'test': 2}
FRACTIONS = {'train': 0.7, 'val': 0.15, 'test': 0.15}
FIELDS = ('frame_id', 'session_id', 'filename', 'received_utc', 'width', 'height', 'tags')
DATA_YAML = 'train: images/train\nval: images/val\ntest: images/test\nnames:\n  0: person\n'
SESSION_PATTERN = re.compile(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}')
FRAME_ID_PATTERN = re.compile(r'[0-9a-f]{32}')
PNG_SIGNATURE = b'\x89PNG\r\n\x1a\n'


class DatasetError(ValueError):
    pass


@dataclass(frozen=True)
class CaptureEntry:
    frame_id: str
    session_id: str
    filename: str
    received_utc: str
    width: int
    height: int
    tags: tuple[str, ...]
    split: str = ''

    def image_path(self, root):
        return root / 'images' / self.split / self.filename

    def label_path(self, root):
        return root / 'labels' / self.split / f'{self.frame_id}.txt'

    def exclusion_path(self, root):
        return root / 'excluded' / f'{self.frame_id}.txt'


def write_atomic(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as stream:
        incomplete = Path(stream.name)
    try:
        incomplete.write_bytes(content)
        os.replace(incomplete, path)
    finally:
        incomplete.unlink(missing_ok=True)


def manifest_bytes(rows, *, frozen=False):
    stream = io.StringIO(newline='')
    writer = csv.writer(stream, lineterminator='\n')
    writer.writerow((*FIELDS, 'split') if frozen else FIELDS)
    for row in rows:
        values = (row.frame_id, row.session_id, row.filename, row.received_utc,
                  row.width, row.height, json.dumps(row.tags, ensure_ascii=False))
        writer.writerow((*values, row.split) if frozen else values)
    return stream.getvalue().encode('utf-8')


def live_frame(frame, now):
    return frame is not None and 0 <= now - frame.received_at <= MAX_FRAME_AGE


class CaptureRun:
    def __init__(self, root, session_id, tags=()):
        if (root / 'manifest.csv').exists():
            raise DatasetError('Capture into the original collection folder or choose a new folder.')
        if not SESSION_PATTERN.fullmatch(session_id):
            raise DatasetError('Session ID must be 1 to 64 letters, digits, hyphens or underscores and start with a letter or digit.')
        if any(not tag.strip() for tag in tags):
            raise DatasetError('Condition tags must contain text.')
        self.root = root
        self.session_id = session_id
        self.tags = tuple(tags)
        self.manifest = root / 'manifests' / f'{uuid4().hex}.csv'
        self.rows = []
        self.seen = set()
        self.last_sequence = -1

    def save(self, frame, *, now):
        if not live_frame(frame, now) or frame.sequence <= self.last_sequence:
            return None
        if frame.received_utc is None or frame.image.shape != (HEIGHT, WIDTH, 3) or frame.image.dtype != np.uint8:
            raise DatasetError('Expected a timestamped native 320x240 color frame.')
        digest = hashlib.sha256(frame.image.tobytes()).digest()
        if digest in self.seen:
            return None
        frame_id = uuid4().hex
        row = CaptureEntry(frame_id, self.session_id, f'{frame_id}.png',
                           frame.received_utc.astimezone(timezone.utc).isoformat(), WIDTH, HEIGHT, self.tags)
        encoded, png = cv2.imencode('.png', frame.image)
        if not encoded:
            raise DatasetError('Could not encode the camera frame as PNG.')
        image_path = row.image_path(self.root)
        write_atomic(image_path, png.tobytes())
        try:
            write_atomic(self.manifest, manifest_bytes([*self.rows, row]))
        except OSError:
            image_path.unlink(missing_ok=True)
            raise
        self.rows.append(row)
        self.seen.add(digest)
        self.last_sequence = frame.sequence
        return row


def read_manifest(root):
    frozen = (root / 'manifest.csv').is_file()
    paths = [root / 'manifest.csv'] if frozen else sorted((root / 'manifests').glob('*.csv'))
    rows = []
    seen = set()
    fields = (*FIELDS, 'split') if frozen else FIELDS
    for path in paths:
        try:
            with path.open(encoding='utf-8', newline='') as stream:
                reader = csv.reader(stream, strict=True)
                if tuple(next(reader, ())) != fields:
                    raise DatasetError(f'Invalid header in {path}')
                for values in reader:
                    try:
                        raw = dict(zip(fields, values, strict=True))
                        tags = json.loads(raw['tags'])
                        received = datetime.fromisoformat(raw['received_utc'])
                        frame_id = raw['frame_id']
                        width, height = int(raw['width']), int(raw['height'])
                        split = raw.get('split', '')
                        if (not FRAME_ID_PATTERN.fullmatch(frame_id) or raw['filename'] != f'{frame_id}.png'
                                or not SESSION_PATTERN.fullmatch(raw['session_id'])
                                or received.utcoffset() != timezone.utc.utcoffset(None)
                                or (width, height) != (WIDTH, HEIGHT) or (frozen and split not in SPLITS)
                                or not isinstance(tags, list)
                                or any(not isinstance(tag, str) or not tag.strip() for tag in tags)):
                            raise ValueError('Invalid frame metadata')
                    except ValueError as exc:
                        raise DatasetError(f'Invalid manifest row at {path}:{reader.line_num}') from exc
                    if frame_id in seen:
                        raise DatasetError(f'Duplicate frame ID {frame_id} in {path}')
                    seen.add(frame_id)
                    rows.append(CaptureEntry(frame_id, raw['session_id'], raw['filename'], raw['received_utc'],
                                             width, height, tuple(tags), split))
        except (csv.Error, UnicodeError) as exc:
            raise DatasetError(f'Cannot read capture manifest {path}') from exc
    if not rows:
        raise DatasetError(f'No captured frames in {root}')
    return sorted(rows, key=lambda row: (datetime.fromisoformat(row.received_utc), row.frame_id))


def read_image(path):
    if not path.is_file():
        raise DatasetError(f'Missing image {path}')
    data = path.read_bytes()
    if not data.startswith(PNG_SIGNATURE):
        raise DatasetError(f'Invalid PNG {path}')
    try:
        image = cv2.imdecode(np.frombuffer(data, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
    except cv2.error as exc:
        raise DatasetError(f'Invalid PNG {path}') from exc
    if image is None or image.shape != (HEIGHT, WIDTH, 3) or image.dtype != np.uint8:
        raise DatasetError(f'Invalid PNG {path}. Expected a 320x240 color image.')
    return image


def write_label(root, row, boxes):
    lines = []
    seen = set()
    for left, top, right, bottom in boxes:
        if not (0 <= left < right <= row.width and 0 <= top < bottom <= row.height):
            raise DatasetError('A person box must have positive area and be inside the frame.')
        values = ((left + right) / (2 * row.width), (top + bottom) / (2 * row.height),
                  (right - left) / row.width, (bottom - top) / row.height)
        if values in seen:
            raise DatasetError('Each person box must be distinct.')
        seen.add(values)
        lines.append('0 ' + ' '.join(format(value, '.12g') for value in values) + '\n')
    write_atomic(row.label_path(root), ''.join(lines).encode('ascii'))
    row.exclusion_path(root).unlink(missing_ok=True)


def exclude_frame(root, row):
    write_atomic(row.exclusion_path(root), b'')


def read_label(path):
    if not path.is_file():
        raise DatasetError(f'Missing label {path}. Confirm empty frames explicitly.')
    boxes = []
    seen = set()
    try:
        for number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            fields = line.split()
            if not fields:
                continue
            try:
                class_id, *coordinates = fields
                x, y, w, h = values = tuple(map(float, coordinates))
                tolerance = 1e-9
                if (class_id != '0' or not all(math.isfinite(value) and 0 <= value <= 1 for value in values)
                        or w <= 0 or h <= 0 or values in seen
                        or min(x - w / 2, y - h / 2) < -tolerance
                        or max(x + w / 2, y + h / 2) > 1 + tolerance):
                    raise ValueError('Invalid class or box')
            except ValueError as exc:
                raise DatasetError(f'Invalid YOLO label at {path}:{number}') from exc
            boxes.append(values)
            seen.add(values)
    except UnicodeError as exc:
        raise DatasetError(f'Invalid YOLO label encoding in {path}') from exc
    return boxes


def check_split_coverage(entries):
    counts = defaultdict(set)
    kinds = defaultdict(set)
    assignments = {}
    for row, positive in entries:
        previous = assignments.setdefault(row.session_id, row.split)
        if previous != row.split:
            raise DatasetError(f'Session {row.session_id} is present in more than one split.')
        counts[row.split].add(row.session_id)
        kinds[row.split].add(positive)
    for split in SPLITS:
        if len(counts[split]) < MIN_SESSIONS[split] or kinds[split] != {True, False}:
            raise DatasetError(f'{split} needs at least {MIN_SESSIONS[split]} distinct session(s) and both positive and negative frames.')


def validate_dataset(root):
    frozen = (root / 'manifest.csv').is_file()
    if frozen:
        manifest_hash = hashlib.sha256((root / 'manifest.csv').read_bytes()).hexdigest()
        checksum = root / 'manifest.sha256'
        if not checksum.is_file() or checksum.read_text(encoding='ascii').strip() != manifest_hash:
            raise DatasetError('The manifest SHA-256 does not match manifest.csv.')
        yaml = root / 'data.yaml'
        if not yaml.is_file() or yaml.read_text(encoding='utf-8') != DATA_YAML:
            raise DatasetError('data.yaml must define class 0 and the three dataset image paths.')
    entries = []
    expected_images = set()
    expected_labels = set()
    digests = {}
    for row in read_manifest(root):
        image_path, label_path = row.image_path(root), row.label_path(root)
        expected_images.add(image_path)
        expected_labels.add(label_path)
        if not frozen and row.exclusion_path(root).is_file():
            continue
        digest = hashlib.sha256(read_image(image_path).tobytes()).digest()
        if digest in digests:
            raise DatasetError(f'Duplicate image pixels in {image_path} and {digests[digest]}.')
        digests[digest] = image_path
        entries.append((row, bool(read_label(label_path))))
    for folder, pattern, expected in (('images', '*', expected_images), ('labels', '*.txt', expected_labels)):
        unexpected = sorted(path for path in (root / folder).rglob(pattern) if path not in expected and path.is_file())
        if unexpected:
            raise DatasetError(f'File without a matching manifest row: {unexpected[0]}')
    if not entries:
        raise DatasetError('No included frames. Label frames to include them in the dataset.')
    if frozen:
        check_split_coverage(entries)
    return entries


def assign_splits(entries, test_sessions):
    counts = Counter(row.session_id for row, _ in entries)
    kinds = defaultdict(set)
    for row, positive in entries:
        kinds[row.session_id].add(positive)
    if len(counts) < sum(MIN_SESSIONS.values()):
        raise DatasetError('Collect at least five independent sessions before exporting.')

    reserved = set(test_sessions)
    if reserved and (len(reserved) < MIN_SESSIONS['test'] or not reserved <= counts.keys()
                     or set().union(*(kinds[session] for session in reserved)) != {True, False}):
        raise DatasetError('Reserved test sessions must include at least two existing sessions and both positive and negative frames.')
    movable = tuple(split for split in SPLITS if split != 'test' or not reserved)
    available = sorted(counts.keys() - reserved, key=lambda session: (-counts[session], session))
    targets = {split: FRACTIONS[split] * len(entries) for split in SPLITS}
    best = None
    best_cost = math.inf
    for use_mixed in product((True, False), repeat=len(movable)):
        remaining = available.copy()
        assignments = dict.fromkeys(reserved, 'test')
        size = Counter({'test': sum(counts[session] for session in reserved)})
        seeds = []
        for split, mixed in zip(movable, use_mixed):
            required = ({True, False},) if mixed else ({True}, {False})
            seeds.extend((split, kind) for kind in required)
        seeded_counts = Counter(assignments.values())
        pending_seeds = Counter(split for split, _ in seeds)
        for split, kind in seeds:
            candidates = [session for session in remaining if kinds[session] == kind]
            if not candidates:
                break
            needed = max(pending_seeds[split], MIN_SESSIONS[split] - seeded_counts[split])
            target = (targets[split] - size[split]) / needed
            chosen = min(candidates, key=lambda session: abs(target - counts[session]))
            remaining.remove(chosen)
            assignments[chosen] = split
            size[split] += counts[chosen]
            seeded_counts[split] += 1
            pending_seeds[split] -= 1
        else:
            extras = [split for split in movable for _ in range(MIN_SESSIONS[split] - seeded_counts[split])]
            if len(remaining) < len(extras):
                continue
            for split in extras:
                chosen = min(remaining, key=lambda session: abs(targets[split] - size[split] - counts[session]))
                remaining.remove(chosen)
                assignments[chosen] = split
                size[split] += counts[chosen]
            for session in remaining:
                split = max(movable, key=lambda split: targets[split] - size[split])
                assignments[session] = split
                size[split] += counts[session]
            cost = sum((size[split] - targets[split]) ** 2 for split in SPLITS)
            if cost < best_cost:
                best, best_cost = assignments, cost
    if best is None:
        raise DatasetError('Cannot give all splits independent sessions and both positive and negative frames. Collect more sessions with both kinds of frames.')
    return best


@dataclass(frozen=True)
class ExportReport:
    counts: dict[str, int]
    manifest_sha256: str
    archive_sha256: str


def publish_new(source, destination):
    # Windows rename and POSIX hard-link both refuse to replace an existing destination.
    if os.name == 'nt':
        source.rename(destination)
    else:
        os.link(source, destination)


def export_dataset(root, output, *, test_sessions=()):
    if (root / 'manifest.csv').exists():
        raise DatasetError('Export from the original capture folder.')
    checksum = output.with_suffix(output.suffix + '.sha256')
    if output.exists() or checksum.exists():
        raise DatasetError(f'Choose a new ZIP name. {output} or its checksum already exists.')
    entries = validate_dataset(root)
    assignments = assign_splits(entries, test_sessions)
    frozen = [(replace(row, split=assignments[row.session_id]), positive) for row, positive in entries]
    check_split_coverage(frozen)
    manifest = manifest_bytes([row for row, _ in frozen], frozen=True)
    manifest_hash = hashlib.sha256(manifest).hexdigest()
    output.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=output.parent) as directory:
        staging = Path(directory)
        archive = staging / 'dataset.zip'
        with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as stream:
            stream.writestr('data.yaml', DATA_YAML)
            stream.writestr('manifest.csv', manifest)
            stream.writestr('manifest.sha256', manifest_hash + '\n')
            for (raw, _), (row, _) in zip(entries, frozen):
                stream.write(raw.image_path(root), row.image_path(Path()).as_posix())
                stream.write(raw.label_path(root), row.label_path(Path()).as_posix())
        extracted = staging / 'extracted'
        with zipfile.ZipFile(archive) as stream:
            stream.extractall(extracted)
        validate_dataset(extracted)
        with archive.open('rb') as stream:
            archive_hash = hashlib.file_digest(stream, 'sha256').hexdigest()
        staged_checksum = staging / 'dataset.sha256'
        write_atomic(staged_checksum, f'{archive_hash}  {output.name}\n'.encode('utf-8'))
        publish_new(staged_checksum, checksum)
        try:
            publish_new(archive, output)
        except OSError:
            checksum.unlink(missing_ok=True)
            raise
    counts = Counter(row.split for row, _ in frozen)
    return ExportReport(dict(counts), manifest_hash, archive_hash)


In [ ]:
%%writefile /content/xiao-phase4/helpers/human_tracker/colab_training.py
"""Train the two YOLO11n recipes on the frozen XIAO study."""

from collections import Counter
import csv
from dataclasses import dataclass
from hashlib import file_digest, sha256
import io
import json
import math
import os
from pathlib import Path
import platform
import random
import shutil
import tempfile
from zipfile import ZipFile

from .dataset import validate_dataset, write_atomic


ULTRALYTICS_VERSION = '8.4.215'
DATASET_SHA256 = '7359ed6192778af8a8795236f9c12b17cf494e2ab21d4d5258b947bcdf2916d2'
MANIFEST_SHA256 = '51023b1bd700c3826a82f2a2e47047eb3c91effc7000313f125bcbd439d66e6e'
WEIGHTS_SHA256 = '0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1'
SEED = 42
IMAGE_SIZE = 320
CONFIDENCE = 0.001
NMS_IOU = 0.7
AUGMENTED_SETTINGS = {'fliplr': 0.50, 'hsv_h': 0.01, 'hsv_s': 0.15, 'hsv_v': 0.20,
                      'translate': 0.05, 'scale': 0.15}
RUN_FIELDS = {'model', 'resume', 'name', 'save_dir', 'data'}


@dataclass(frozen=True)
class StudyData:
    root: Path
    entries: tuple
    holdout_fingerprint: dict[str, str]

    @property
    def yaml(self):
        return self.root / 'data.yaml'


def digest(path):
    with Path(path).open('rb') as stream:
        return file_digest(stream, 'sha256').hexdigest()


def heldout_digests(root, entries):
    hashes = {split: sha256() for split in ('val', 'test')}
    for row, _ in entries:
        if row.split in hashes:
            for path in (row.image_path(root), row.label_path(root)):
                hashes[row.split].update(path.relative_to(root).as_posix().encode('utf-8'))
                hashes[row.split].update(bytes.fromhex(digest(path)))
    return {split: value.hexdigest() for split, value in hashes.items()}


def prepare_dataset(archive, work):
    if digest(archive) != DATASET_SHA256:
        raise ValueError('Dataset ZIP SHA-256 differs from the frozen xiao-v1.zip.')
    work = Path(work).resolve()
    work.mkdir(parents=True, exist_ok=True)
    root = Path(tempfile.mkdtemp(dir=work, prefix='xiao-v1-'))
    with ZipFile(archive) as zipped:
        zipped.extractall(root)
    if digest(root / 'manifest.csv') != MANIFEST_SHA256:
        raise ValueError('Manifest SHA-256 differs from the frozen study.')
    entries = tuple(validate_dataset(root))
    counts = Counter(row.split for row, _ in entries)
    if counts != {'train': 240, 'val': 60, 'test': 60}:
        raise ValueError(f'Unexpected frozen split counts {counts}.')
    return StudyData(root, entries, heldout_digests(root, entries))


def check_holdouts(study):
    if (digest(study.root / 'manifest.csv') != MANIFEST_SHA256
            or heldout_digests(study.root, study.entries) != study.holdout_fingerprint):
        raise ValueError('The frozen manifest or held-out image and label bytes changed.')


def make_smoke_data(study):
    """Use deterministic lists of training and validation images, including negatives."""
    rng = random.Random(SEED)
    config = {'path': str(study.root), 'names': {0: 'person'}}
    for split, positives in (('train', 14), ('val', 6)):
        files = []
        for status, count in ((True, positives), (False, 2)):
            pool = [row.image_path(study.root) for row, present in study.entries
                    if row.split == split and present == status]
            files.extend(rng.sample(pool, count))
        rng.shuffle(files)
        file_list = study.root / f'smoke-{split}.txt'
        file_list.write_text(''.join(path.as_posix() + '\n' for path in files), encoding='utf-8')
        config[split] = file_list.as_posix()
    target = study.root / 'smoke.yaml'
    target.write_text(json.dumps(config, indent=2), encoding='utf-8')
    return target


def recipe(name, *, batch=8, epochs=40):
    if name not in ('A', 'B') or not isinstance(batch, int) or batch < 1 or not 1 <= epochs <= 40:
        raise ValueError('Choose recipe A or B, a positive batch, and 1 to 40 epochs.')
    transforms = AUGMENTED_SETTINGS if name == 'B' else dict.fromkeys(AUGMENTED_SETTINGS, 0.0)
    return {
        'seed': SEED, 'imgsz': IMAGE_SIZE, 'batch': batch, 'epochs': epochs, 'patience': 10,
        'optimizer': 'AdamW', 'lr0': 0.002, 'momentum': 0.9, 'warmup_bias_lr': 0.0,
        'deterministic': True, 'workers': 0, 'amp': False, 'cache': False, 'multi_scale': 0.0,
        'val': True, 'plots': True, 'augment': False, 'save': True, 'pretrained': True,
        'conf': CONFIDENCE, 'iou': NMS_IOU, 'rect': False,
        'mosaic': 0.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 0,
        'degrees': 0.0, 'perspective': 0.0, 'shear': 0.0, 'flipud': 0.0, 'bgr': 0.0,
        'augmentations': [], **transforms,
    }


def verify_recipes(plain, augmented):
    for key in (plain.keys() | augmented.keys() | AUGMENTED_SETTINGS.keys()) - RUN_FIELDS:
        if key in AUGMENTED_SETTINGS:
            if plain.get(key) != 0.0 or augmented.get(key) != AUGMENTED_SETTINGS[key]:
                raise ValueError(f'Unexpected augmentation setting {key}.')
        elif key not in plain or key not in augmented or plain[key] != augmented[key]:
            raise ValueError(f'Recipes must share setting {key}.')


def runtime_info(device):
    import cv2
    import numpy as np
    import torch
    import ultralytics

    if ultralytics.__version__ != ULTRALYTICS_VERSION:
        raise RuntimeError(f'Install ultralytics=={ULTRALYTICS_VERSION} and restart the kernel.')
    for key in ('sync', 'wandb', 'mlflow', 'comet', 'neptune', 'clearml', 'tensorboard', 'dvc', 'raytune'):
        if key in ultralytics.settings and ultralytics.settings[key]:
            ultralytics.settings.update({key: False})
    return {'python': platform.python_version(), 'torch': str(torch.__version__), 'cuda': torch.version.cuda,
            'ultralytics': ultralytics.__version__, 'opencv': cv2.__version__, 'numpy': np.__version__,
            'device': str(device), 'gpu': torch.cuda.get_device_name(device) if isinstance(device, int) else None}


def verify_initial_weights(path):
    path = Path(path).resolve()
    if digest(path) != WEIGHTS_SHA256:
        raise ValueError('Use the same official yolo11n.pt from Human_Tracker/models.')
    return path


def save_json(path, value):
    write_atomic(path, (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8'))


def validate_model(weights, data_yaml, run_dir, device, batch):
    from ultralytics import YOLO

    model = YOLO(str(weights))
    if model.names.get(0) != 'person':
        raise ValueError(f'Expected person class 0 in {weights}.')
    results = model.val(
        data=str(data_yaml), split='val', classes=[0], imgsz=IMAGE_SIZE, batch=batch,
        conf=CONFIDENCE, iou=NMS_IOU, device=device, workers=0, augment=False, plots=False,
        project=str(run_dir.parent), name=run_dir.name, exist_ok=True,
    )
    metrics = {key: float(value) for key, value in results.results_dict.items()}
    if not metrics or not all(math.isfinite(value) for value in metrics.values()):
        raise ValueError('Validation returned missing or non-finite metrics.')
    return metrics


def baseline(study, weights, output, device, batch):
    versions = runtime_info(device)
    weights = verify_initial_weights(weights)
    metrics = validate_model(weights, study.yaml, output / 'baseline-val', device, batch)
    check_holdouts(study)
    save_json(output / 'baseline-val.json', {
        'dataset_sha256': DATASET_SHA256, 'manifest_sha256': MANIFEST_SHA256,
        'checkpoint_sha256': WEIGHTS_SHA256, 'split': 'val', 'class_id': 0, 'seed': SEED,
        'imgsz': IMAGE_SIZE, 'batch': batch, 'conf': CONFIDENCE, 'iou': NMS_IOU,
        'versions': versions, 'metrics': metrics,
    })
    return metrics


def persist_checkpoint(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=target.parent, delete=False) as stream:
        staged = Path(stream.name)
    try:
        shutil.copyfile(source, staged)
        os.replace(staged, target)
    finally:
        staged.unlink(missing_ok=True)


def saved_bests(folder):
    for path in folder.glob('best-epoch-*.pt'):
        epoch = path.stem.removeprefix('best-epoch-')
        if epoch.isdecimal():
            yield int(epoch), path


def restore_recovery(best, history, recovery, completed, results):
    """Restore the best weights and CSV that belong to the committed last.pt."""
    candidates = dict(saved_bests(recovery))
    best_epoch = max((epoch for epoch in candidates if epoch <= completed), default=0)
    if not best_epoch or results.get('epoch') != list(range(1, completed + 1)):
        raise ValueError(f'The recovery state in {recovery} is incomplete.')
    stream = io.StringIO(newline='')
    writer = csv.writer(stream, lineterminator='\n')
    writer.writerow(results)
    writer.writerows(zip(*results.values(), strict=True))
    persist_checkpoint(candidates[best_epoch], best)
    write_atomic(history, stream.getvalue().encode('utf-8'))
    for epoch, path in candidates.items():
        if epoch > completed:
            path.unlink()


def require_matching_batch(trainer, requested_batch):
    if trainer.batch_size != requested_batch:
        raise RuntimeError(f'Training switched to batch size {trainer.batch_size}. '
                           'Choose a smaller shared BATCH and a fresh RUN_GROUP for A and B.')


class PauseAfterCheckpoint(Exception):
    pass


def train_recipe(study, weights, output, name, device, *, batch=8, smoke=False, pause_after=None):
    from ultralytics import YOLO

    versions = runtime_info(device)
    weights = verify_initial_weights(weights)
    epochs = (1 if name == 'A' else 2) if smoke else 40
    config = recipe(name, batch=batch, epochs=epochs)
    if smoke and batch > 16:
        raise ValueError('Set BATCH at or below 16 for the smoke dataset.')
    data_yaml = make_smoke_data(study) if smoke else study.yaml
    run = output / (f'smoke-{name}' if smoke else name)
    best = run / 'weights' / 'best.pt'
    recovery = run / 'recovery'
    last = recovery / 'last.pt'
    history_path = run / 'results.csv'
    provenance = run / 'experiment.json'
    summary = run / 'summary.json'
    experiment = {'dataset_sha256': DATASET_SHA256, 'manifest_sha256': MANIFEST_SHA256,
                  'initial_checkpoint_sha256': WEIGHTS_SHA256, 'ultralytics': ULTRALYTICS_VERSION,
                  'recipe': name, 'smoke': smoke, 'settings': config}
    if provenance.is_file():
        if json.loads(provenance.read_text(encoding='utf-8')) != experiment:
            raise ValueError(f'{run} uses different settings. Choose a new experiment folder.')
    else:
        save_json(provenance, experiment)
    if summary.is_file():
        report = json.loads(summary.read_text(encoding='utf-8'))
        if (digest(best) != report['best_sha256'] or digest(last) != report['last_sha256']
                or digest(run / 'args.yaml') != report['config_sha256']):
            raise ValueError(f'Checkpoint or config checksum mismatch in {run}.')
        return report

    resume_epoch, early_stopped = 0, False
    if last.is_file():
        model = YOLO(str(last))
        state = model.ckpt
        resume_epoch = int(state['epoch']) + 1
        if (not state.get('optimizer') or not 0 < resume_epoch <= epochs
                or state['train_args']['batch'] != batch):
            raise ValueError(f'Expected a resumable checkpoint with batch {batch} at {last}.')
        restore_recovery(best, history_path, recovery, resume_epoch, state['train_results'])
        early_stopped = resume_epoch - state['stopper']['best_epoch'] >= config['patience']
    else:
        history_path.unlink(missing_ok=True)
        for _, path in saved_bests(recovery):
            path.unlink()
        model = YOLO(str(weights))

    def audit_start(trainer):
        expected_device = 'cuda' if isinstance(device, int) else 'cpu'
        if trainer.start_epoch != resume_epoch or trainer.device.type != expected_device:
            raise RuntimeError('Training started on an unexpected device or epoch.')
        for transform in trainer.train_loader.dataset.transforms.transforms:
            if type(transform).__name__ == 'Albumentations':
                pipeline = transform.transform
                if pipeline is not None and pipeline.transforms:
                    raise RuntimeError('Unexpected Albumentations transforms in the training loader.')

    def save_epoch(trainer):
        require_matching_batch(trainer, batch)
        completed = trainer.epoch + 1
        snapshot = None
        if trainer.best_fitness == trainer.fitness:
            snapshot = recovery / f'best-epoch-{completed}.pt'
            persist_checkpoint(trainer.best, snapshot)
        persist_checkpoint(trainer.last, last)
        if snapshot is not None:
            for epoch, path in saved_bests(recovery):
                if epoch < completed:
                    path.unlink()
        if pause_after == completed:
            raise PauseAfterCheckpoint

    if resume_epoch < epochs and not early_stopped:
        model.add_callback('on_train_start', audit_start)
        model.add_callback('on_train_epoch_start', lambda trainer: require_matching_batch(trainer, batch))
        model.add_callback('on_model_save', save_epoch)
        try:
            if resume_epoch:
                model.train(resume=True, data=str(data_yaml), device=device, save_dir=str(run))
            else:
                model.train(data=str(data_yaml), project=str(output), name=run.name,
                            exist_ok=True, device=device, **config)
        except PauseAfterCheckpoint:
            return {'paused_after_epoch': pause_after, 'last_sha256': digest(last)}

    with history_path.open(newline='', encoding='utf-8') as stream:
        history = list(csv.DictReader(stream))
    if not history or not all(math.isfinite(float(value)) for row in history for value in row.values()):
        raise ValueError(f'Missing or non-finite training results in {history_path}.')
    completed = int(float(history[-1]['epoch']))
    if [int(float(row['epoch'])) for row in history] != list(range(1, completed + 1)):
        raise ValueError(f'Training results contain missing or repeated epochs in {history_path}.')
    best_model = YOLO(str(best))
    if best_model.names != {0: 'person'}:
        raise ValueError(f'{best} must have exactly one class, person.')
    metrics = validate_model(best, data_yaml, run / 'best-val', device, batch)
    check_holdouts(study)
    report = {'epochs_completed': completed, 'resumed_from_epoch': resume_epoch,
              'best_sha256': digest(best), 'last_sha256': digest(last),
              'versions': versions, 'metrics': metrics, 'config_sha256': digest(run / 'args.yaml'),
              'dataset_sha256': DATASET_SHA256, 'manifest_sha256': MANIFEST_SHA256}
    save_json(summary, report)
    return report


def smoke_check(study, weights, output, device, batch):
    from ultralytics import YOLO

    plain = train_recipe(study, weights, output, 'A', device, batch=batch, smoke=True)
    resumed = train_recipe(study, weights, output, 'B', device, batch=batch, smoke=True, pause_after=1)
    if 'paused_after_epoch' in resumed:
        resumed = train_recipe(study, weights, output, 'B', device, batch=batch, smoke=True)
    if plain['epochs_completed'] != 1 or resumed['epochs_completed'] != 2 or resumed['resumed_from_epoch'] != 1:
        raise RuntimeError('Expected a one-epoch run and a two-epoch run resumed after epoch one.')
    image = next(row.image_path(study.root) for row, person in study.entries if row.split == 'val' and person)
    prediction = YOLO(str(output / 'smoke-B' / 'weights' / 'best.pt')).predict(
        str(image), device=device, imgsz=IMAGE_SIZE, conf=CONFIDENCE, verbose=False,
    )
    if not prediction or prediction[0].orig_shape != (240, 320):
        raise RuntimeError('The smoke export did not return a prediction for the validation image.')
    return {'A': plain, 'B': resumed, 'prediction_boxes': len(prediction[0].boxes)}


## Verify CUDA and the frozen study

The notebook checks the archive, manifest, image-label pairs, split membership, and starting checkpoint.


In [ ]:
import json
from urllib.request import urlopen
import torch
from human_tracker.colab_training import (
    baseline, check_holdouts, prepare_dataset, recipe, runtime_info, smoke_check,
    train_recipe, verify_initial_weights, verify_recipes,
)
from human_tracker.dataset import write_atomic

device = None
if torch.cuda.is_available():
    try:
        torch.empty(1, device='cuda:0')
        device = 0
    except RuntimeError as error:
        print(f'CUDA allocation failed. Use a fresh GPU runtime. {error}')

print(json.dumps(runtime_info(device if device is not None else 'cpu'), indent=2))
archive = inputs / 'xiao-v1.zip'
if not archive.is_file():
    raise FileNotFoundError(f'Upload xiao-v1.zip to {inputs} and run this cell again.')
study = prepare_dataset(archive, work)
checkpoint = inputs / 'yolo11n.pt'
if not checkpoint.is_file():
    checkpoint = work / 'yolo11n.pt'
    if not checkpoint.is_file():
        url = 'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt'
        with urlopen(url, timeout=60) as response:
            write_atomic(checkpoint, response.read())
verify_initial_weights(checkpoint)
verify_recipes(recipe('A', batch=BATCH), recipe('B', batch=BATCH))
print(f'Verified {len(study.entries)} frames and the original checkpoint.')
if device is None:
    print('Use a GPU runtime to run the smoke checks and full recipes.')


## Pretrained baseline on validation

Evaluate the original COCO person class at 320 pixels with a confidence floor of 0.001 and NMS IoU of 0.70.


In [ ]:
baseline_metrics = baseline(study, checkpoint, output, device if device is not None else 'cpu', BATCH)
print(json.dumps(baseline_metrics, indent=2))


## Smoke and resume on a small subset

A trains for one epoch on 16 training and 8 validation images. B trains for two epochs and pauses after the first saved epoch. The check reloads the checkpoint, verifies that training restarts at epoch two, checks finite results and loads the exported checkpoint for prediction.

Inspect the boxes in both batches below. In B, the boxes should follow people after flipping, shifting or resizing.


In [ ]:
from IPython.display import Image, display

if device is None:
    print('Select a GPU runtime and rerun setup to run the smoke checks.')
else:
    smoke_results = smoke_check(study, checkpoint, output, device, BATCH)
    print(json.dumps(smoke_results, indent=2))
    for name in ('A', 'B'):
        batch_plot = output / f'smoke-{name}' / 'train_batch0.jpg'
        display(Image(filename=str(batch_plot)))


## Train both full recipes

Both use seed 42, early-stopping patience 10 and up to 40 epochs. If GPU memory requires a smaller batch, set the same smaller **BATCH** for both, choose a fresh **RUN_GROUP**, and repeat the smoke checks.

After a disconnect, rerun setup with the same paths, **RUN_GROUP**, and **BATCH**. The affected training cell resumes from the last saved epoch.


In [ ]:
# @title Recipe A (stochastic augmentation disabled)
if device is None:
    print('Select a GPU runtime before training.')
else:
    report = train_recipe(study, checkpoint, output, 'A', device, batch=BATCH)
    print(json.dumps(report, indent=2))


In [ ]:
# @title Recipe B (modest training augmentation)
if device is None:
    print('Select a GPU runtime before training.')
else:
    report = train_recipe(study, checkpoint, output, 'B', device, batch=BATCH)
    print(json.dumps(report, indent=2))


## Verify the runs and locate the Phase 5 files

Each run saves its Ultralytics configuration and validation metrics. The weights/best.pt file is the candidate for Phase 5. The recovery/last.pt file retains optimizer state.


In [ ]:
import yaml

if device is not None:
    configs = [yaml.safe_load((output / name / 'args.yaml').read_text(encoding='utf-8')) for name in ('A', 'B')]
    verify_recipes(*configs)
    check_holdouts(study)
    for name in ('A', 'B'):
        run = output / name
        print(f'{name} model  {run / "weights" / "best.pt"}')
        print(f'{name} results  {run / "summary.json"}')
    print(f'Baseline results  {output / "baseline-val.json"}')
